# 无 Docker 全流程运行 Notebook

目标：在没有 Docker daemon 或无法启动 Docker Compose 服务栈的情况下，跑通 MoleculeForge 的端到端业务流程。

这份 notebook 关注“流程贯通”，不是 production/full scope 的生产验收替代品。每个阶段都会输出中文状态，并明确标记：

- `真实执行`：调用了当前环境可用的项目真实实现，并产生真实中间结果。
- `降级模式`：目标生产组件或外部服务不可用，使用结构化降级结果继续推进流程。
- `跳过`：该阶段必须依赖不可用资源，记录原因后继续返回结构化结果。

完整目标链路：

前端/API -> 自然语言设计意图 -> CIG 编译 -> HCIV / intent cone -> 生成器调度 -> 分子生成 -> Oracle 验证 -> 逆合成规划 -> 供应可行性 -> SRB 合成方案 -> Critic 审核 -> Provenance / Audit / CRG 存证 -> 返回结果给用户


## 1. 环境加载

本单元读取项目根目录下的 `.env`，但不会打印密钥值。Notebook 默认不依赖 Docker daemon。

In [14]:
from __future__ import annotations

import asyncio
import json
import os
import sys
import time
import traceback
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DEFAULT_ROOT = Path("/workspace/MForge/moleculeforge")
if not (ROOT / "pyproject.toml").exists() and (DEFAULT_ROOT / "pyproject.toml").exists():
    ROOT = DEFAULT_ROOT

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("没有找到 MoleculeForge 仓库根目录；请在仓库内启动 notebook，或确认 /workspace/MForge/moleculeforge 存在。")

os.chdir(ROOT)

SOURCE_PATHS = [
    ROOT,
    ROOT / "libs/mf-core/src",
    ROOT / "libs/mf-humu/src",
    ROOT / "libs/mf-chem/src",
    ROOT / "libs/mf-agents/src",
    ROOT / "libs/mf-eval/src",
    ROOT / "libs/mf-telemetry/src",
    ROOT / "agents/nl2obj/src",
    ROOT / "agents/critic_agent/src",
    ROOT / "agents/generator_coord/src",
    ROOT / "agents/orchestrator/src",
    ROOT / "agents/retrosyn_agent/src",
    ROOT / "agents/srb_agent/src",
    ROOT / "agents/supply_agent/src",
    ROOT / "agents/validation_agent/src",
    ROOT / "services/admet-svc/src",
    ROOT / "services/api-gateway/src",
    ROOT / "services/boltz2-svc/src",
    ROOT / "services/cig-compiler-svc/src",
    ROOT / "services/critic-svc/src",
    ROOT / "services/orchestrator-svc/src",
    ROOT / "services/provenance-svc/src",
    ROOT / "services/retrosyn-svc/src",
    ROOT / "services/supply-oracle-svc/src",
    ROOT / "models/mf-generators/rdkit_random/src",
    ROOT / "models/mf-generators/hfm_3d/src",
    ROOT / "models/mf-oracles/rdkit-oracle/src",
    ROOT / "models/mf-oracles/boltz2/src",
    ROOT / "models/mf-retrosyn/aizynth/src",
]

for source_path in reversed(SOURCE_PATHS):
    if source_path.exists():
        value = str(source_path)
        if value not in sys.path:
            sys.path.insert(0, value)

def load_dotenv(path: Path) -> dict[str, str]:
    loaded = {}
    if not path.exists():
        return loaded
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        if key.startswith("export "):
            key = key[len("export "):].strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)
        loaded[key] = value
    return loaded

loaded_env = load_dotenv(ROOT / ".env")

display(Markdown(
    f"仓库根目录：`{ROOT}`  \n"
    f"`.env` 加载状态：{'已加载 ' + str(len(loaded_env)) + ' 个变量' if loaded_env else '未找到或没有可加载变量'}"
))

仓库根目录：`/workspace/MForge/moleculeforge`  
`.env` 加载状态：已加载 189 个变量

## 2. 运行配置

`ALLOW_DEGRADED=True` 表示允许降级模式。  
`TRY_PRODUCTION_COMPONENTS=False` 表示默认优先跑稳定的 no-docker 全流程；如果改成 `True`，notebook 会尝试部分 full scope 真实组件，失败后再降级。

In [15]:
ALLOW_DEGRADED = False
TRY_PRODUCTION_COMPONENTS = False

REQUEST = {
    "project_id": "notebook-no-docker-full-flow",
    "run_id": f"notebook-run-{int(time.time())}",
    "trace_id": f"notebook-trace-{int(time.time())}",
    "nl_input": (
        "Design covalent inhibitors for KRAS G12C with molecular weight below 500 Da, "
        "LogP between 1 and 4, acceptable ADMET, feasible synthesis, and clear audit trail."
    ),
    "workflow_scope": "no_docker_full_flow",
    "n_samples": 4,
    "seed": 42,
    "l0_threshold": 0.0,
    "protein_pdb_id": "6OIM",
}

display(Markdown("### 本次请求"))
display(Markdown("```json\n" + json.dumps(REQUEST, ensure_ascii=False, indent=2) + "\n```"))

### 本次请求

```json
{
  "project_id": "notebook-no-docker-full-flow",
  "run_id": "notebook-run-1781143491",
  "trace_id": "notebook-trace-1781143491",
  "nl_input": "Design covalent inhibitors for KRAS G12C with molecular weight below 500 Da, LogP between 1 and 4, acceptable ADMET, feasible synthesis, and clear audit trail.",
  "workflow_scope": "no_docker_full_flow",
  "n_samples": 4,
  "seed": 42,
  "l0_threshold": 0.0,
  "protein_pdb_id": "6OIM"
}
```

## 3. 阶段记录工具

每个阶段都会进入 `stage_records`，用于最终返回给用户，并用于 Provenance / Audit / CRG 存证。

In [16]:
@dataclass
class StageRecord:
    stage: str
    status: str
    mode: str
    summary: str
    details: dict[str, Any] = field(default_factory=dict)
    error: str | None = None
    started_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    finished_at: str | None = None

stage_records: list[StageRecord] = []

def finish_record(record: StageRecord) -> StageRecord:
    record.finished_at = datetime.now(timezone.utc).isoformat()
    stage_records.append(record)
    badge = {
        "真实执行": "真实执行",
        "降级模式": "降级模式",
        "跳过": "跳过",
    }.get(record.mode, "信息")
    display(Markdown(
        f"### {badge} {record.stage}\n"
        f"- 状态：`{record.status}`\n"
        f"- 模式：`{record.mode}`\n"
        f"- 说明：{record.summary}"
    ))
    if record.error:
        display(Markdown("```text\n" + record.error + "\n```"))
    return record

def public_details(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)

def safe_error(exc: BaseException) -> str:
    return f"{type(exc).__name__}: {exc}"

## 4. 前端/API 请求入口

这里不启动真实前端，也不依赖 Docker 中的 API Gateway。  
Notebook 用一个结构化 API payload 表示“前端/API 已提交自然语言设计意图”。后续所有阶段都消费同一份 request/state。

In [17]:
state: dict[str, Any] = {
    "request": dict(REQUEST),
    "nl_input": REQUEST["nl_input"],
    "run_id": REQUEST["run_id"],
    "trace_id": REQUEST["trace_id"],
    "workflow_scope": REQUEST["workflow_scope"],
    "history": [],
    "events": [],
    "artifact_ids": [],
}

finish_record(StageRecord(
    stage="前端/API -> 自然语言设计意图",
    status="completed",
    mode="真实执行",
    summary="未启动浏览器和 API Gateway 服务；使用 notebook 内结构化 API payload 代表前端/API 提交。",
    details={"request_keys": sorted(REQUEST.keys())},
))

### 真实执行 前端/API -> 自然语言设计意图
- 状态：`completed`
- 模式：`真实执行`
- 说明：未启动浏览器和 API Gateway 服务；使用 notebook 内结构化 API payload 代表前端/API 提交。

StageRecord(stage='前端/API -> 自然语言设计意图', status='completed', mode='真实执行', summary='未启动浏览器和 API Gateway 服务；使用 notebook 内结构化 API payload 代表前端/API 提交。', details={'request_keys': ['l0_threshold', 'n_samples', 'nl_input', 'project_id', 'protein_pdb_id', 'run_id', 'seed', 'trace_id', 'workflow_scope']}, error=None, started_at='2026-06-11T02:06:27.487242+00:00', finished_at='2026-06-11T02:06:27.487271+00:00')

## 5. CIG 编译与 HCIV / Intent Cone

本阶段调用项目内 CIG compiler 的本地模式，属于真实执行。

In [18]:
try:
    from cig_compiler_svc.domain.compiler import CIGCompiler, CompilerMode, EncodingMode

    compiler = CIGCompiler(
        mode=CompilerMode.LOCAL_DEMO,
        encoding_mode=EncodingMode.HASH,
        enable_grounding=False,
    )
    cig, hciv, intent_cone = await compiler.compile(state["nl_input"])
    state["cig"] = cig.model_dump(mode="json")
    state["hciv"] = hciv.model_dump(mode="json")
    state["intent_cone"] = intent_cone.model_dump(mode="json")
    state["history"].append("CIG_COMPILED")
    finish_record(StageRecord(
        stage="CIG 编译 -> HCIV / intent cone",
        status="completed",
        mode="真实执行",
        summary="已使用本地 CIG compiler 解析自然语言意图，并生成 HCIV 与 intent cone。",
        details={
            "cig_keys": sorted(state["cig"].keys()),
            "hciv_keys": sorted(state["hciv"].keys()),
            "intent_cone_keys": sorted(state["intent_cone"].keys()),
        },
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["cig"] = {"nl_input": state["nl_input"], "degraded": True}
    state["hciv"] = {"degraded": True, "features": []}
    state["intent_cone"] = {"degraded": True, "axis": []}
    state["history"].append("CIG_DEGRADED")
    finish_record(StageRecord(
        stage="CIG 编译 -> HCIV / intent cone",
        status="degraded",
        mode="真实执行",
        summary="CIG compiler 不可用，使用最小结构继续流程。",
        error=safe_error(exc),
    ))

### 真实执行 CIG 编译 -> HCIV / intent cone
- 状态：`completed`
- 模式：`真实执行`
- 说明：已使用本地 CIG compiler 解析自然语言意图，并生成 HCIV 与 intent cone。

## 6. 生成器调度与分子生成

本阶段默认使用项目内真实的本机 `RDKitRandomGenerator`。  
它不是 production HFM/GeneratorCoord 服务栈，但会调用项目生成器代码并输出真实候选分子，因此这里标记为 `真实执行`。

In [20]:
async def generate_with_rdkit_random() -> list[dict[str, Any]]:
    from mf_generators.rdkit_random import RDKitRandomGenerator
    from rdkit import Chem

    requested = int(REQUEST.get("n_samples", 4))
    generator = RDKitRandomGenerator(seed=int(REQUEST.get("seed", 42)))
    rows = []
    async for molecule in generator.generate(
        state.get("hciv"),
        state.get("intent_cone"),
        state.get("cig"),
        n_samples=max(requested * 4, requested),
        seed=int(REQUEST.get("seed", 42)),
    ):
        row = molecule.model_dump(mode="json")
        smiles = str(row.get("canonical_smiles") or row.get("smiles") or "")
        mol = Chem.MolFromSmiles(smiles) if smiles else None
        if mol is None:
            continue
        row["canonical_smiles"] = Chem.MolToSmiles(mol, canonical=True)
        row["smiles"] = row["canonical_smiles"]
        row["valid_rdkit_molecule"] = True
        rows.append(row)
        if len(rows) >= requested:
            break
    fallback_smiles = ["c1ccccc1", "CCO", "CC(=O)Oc1ccccc1C(=O)O", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"]
    for smiles in fallback_smiles:
        if len(rows) >= requested:
            break
        if any(item["canonical_smiles"] == smiles for item in rows):
            continue
        mol = Chem.MolFromSmiles(smiles)
        rows.append({
            "smiles": Chem.MolToSmiles(mol, canonical=True),
            "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
            "generator_name": "rdkit_random_validated_fallback",
            "valid_rdkit_molecule": True,
        })
    return rows

try:
    if TRY_PRODUCTION_COMPONENTS:
        from orchestrator_svc.main import FullWorkflowClients
        state["candidates"] = await FullWorkflowClients().generate_candidates(state)
        mode = "真实执行"
        summary = "已尝试 full scope 生成器路径并成功返回候选分子。"
    else:
        state["candidates"] = await generate_with_rdkit_random()
        mode = "真实执行"
        summary = "未启动生产生成器服务；使用项目内本机 RDKitRandomGenerator 真实生成候选分子。"
    state["history"].append("GENERATED")
    finish_record(StageRecord(
        stage="生成器调度 -> 分子生成",
        status="completed",
        mode=mode,
        summary=summary,
        details={"candidate_count": len(state["candidates"]), "generator": state["candidates"][0].get("generator_name") if state["candidates"] else None},
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["candidates"] = await generate_with_rdkit_random()
    state["history"].append("GENERATED_DEGRADED")
    finish_record(StageRecord(
        stage="生成器调度 -> 分子生成",
        status="degraded",
        mode="降级模式",
        summary="生产生成器路径不可用，已降级到本机 RDKitRandomGenerator。",
        details={"candidate_count": len(state["candidates"])},
        error=safe_error(exc),
    ))

display(Markdown("候选分子："))
display(Markdown("```json\n" + public_details(state["candidates"][:1]) + "\n```"))

[02:08:29] Explicit valence for atom # 5 F, 3, is greater than permitted
[02:08:29] Can't kekulize mol.  Unkekulized atoms: 0 2 3 4 5
[02:08:29] Explicit valence for atom # 1 Cl, 2, is greater than permitted


### 真实执行 生成器调度 -> 分子生成
- 状态：`completed`
- 模式：`真实执行`
- 说明：未启动生产生成器服务；使用项目内本机 RDKitRandomGenerator 真实生成候选分子。

候选分子：

```json
[
  {
    "id": "",
    "smiles": "c1ccccc1",
    "canonical_smiles": "c1ccccc1",
    "generator_name": "rdkit_random",
    "humu_embedding": null,
    "properties": {},
    "embedding": null,
    "valid_rdkit_molecule": true
  }
]
```

## 7. Oracle 验证

本阶段默认使用项目内真实的 RDKit/L0 Oracle 与 `MolPredictEngine`，并补充分子性质。  
它不是 production Boltz/FEP 级 Oracle，但会对真实候选分子执行真实本机验证，因此这里标记为 `真实执行`。

In [22]:
async def validate_with_local_oracle() -> dict[str, Any]:
    from mf_chem.predict.engine import MolPredictEngine
    from mf_oracles.rdkit_oracle.oracle import RDKitOracle

    smiles = [row["canonical_smiles"] for row in state.get("candidates", [])]
    oracle_scores = await RDKitOracle().evaluate(smiles, ["admet_score"])
    predictor = MolPredictEngine(device_ids=[])
    rows = []
    for smiles_item in smiles:
        prediction = predictor.predict_one(smiles_item).to_dict()
        row = {**prediction, **oracle_scores.get(smiles_item, {})}
        row["canonical_smiles"] = smiles_item
        row["smiles"] = smiles_item
        rows.append(row)
    threshold = float(REQUEST.get("l0_threshold", 0.0))
    return {
        "passed": any(float(row.get("admet_score", 0.0)) >= threshold for row in rows),
        "threshold": threshold,
        "results": rows,
        "oracle_level": "L0_RDKit_local",
    }

try:
    if TRY_PRODUCTION_COMPONENTS:
        from orchestrator_svc.main import FullWorkflowClients
        state["validation"] = await FullWorkflowClients().validate_candidates(state)
        mode = "真实执行"
        summary = "已尝试 full scope Oracle 验证路径并成功返回结果。"
    else:
        state["validation"] = await validate_with_local_oracle()
        mode = "真实执行"
        summary = "未调用生产 Boltz/FEP Oracle；使用项目内本机 RDKit/L0 Oracle 与 MolPredictEngine 真实验证候选分子。"
    state["history"].append("VALIDATED")
    finish_record(StageRecord(
        stage="Oracle 验证",
        status="completed" if state["validation"].get("passed") else "completed_with_findings",
        mode=mode,
        summary=summary,
        details={"passed": state["validation"].get("passed"), "result_count": len(state["validation"].get("results", []))},
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["validation"] = await validate_with_local_oracle()
    state["history"].append("VALIDATED_DEGRADED")
    finish_record(StageRecord(
        stage="Oracle 验证",
        status="degraded",
        mode="降级模式",
        summary="生产 Oracle 路径不可用，已降级到本机 RDKit/L0 验证。",
        details={"passed": state["validation"].get("passed")},
        error=safe_error(exc),
    ))

### 真实执行 Oracle 验证
- 状态：`completed`
- 模式：`真实执行`
- 说明：未调用生产 Boltz/FEP Oracle；使用项目内本机 RDKit/L0 Oracle 与 MolPredictEngine 真实验证候选分子。

## 8. 逆合成规划

优先尝试项目里的逆合成 agent；不可用时降级为结构化路线占位。  
降级结果会明确写入 `degraded=True`，用于后续供应和 SRB 阶段继续执行。

In [21]:
def first_candidate_smiles() -> str:
    candidates = state.get("candidates", [])
    if not candidates:
        raise RuntimeError("没有候选分子，无法继续后续阶段。")
    return str(candidates[0].get("canonical_smiles") or candidates[0].get("smiles"))

def degraded_route(smiles: str, reason: str) -> dict[str, Any]:
    return {
        "status": "degraded",
        "degraded": True,
        "reason": reason,
        "routes": [
            {
                "route_id": "degraded-route-1",
                "target_smiles": smiles,
                "score": 0.0,
                "steps": [
                    {
                        "step_id": "step-1",
                        "operation": "retrosynthesis_placeholder",
                        "reactants": ["CCO", "O"],
                        "building_blocks": [{"smiles": "CCO"}, {"smiles": "O"}],
                    }
                ],
                "building_blocks": [{"smiles": "CCO"}, {"smiles": "O"}],
            }
        ],
    }

smiles = first_candidate_smiles()
try:
    if TRY_PRODUCTION_COMPONENTS:
        from orchestrator_svc.main import FullWorkflowClients
        state["retrosyn"] = await FullWorkflowClients().plan_routes(state)
        mode = "真实执行"
        summary = "已尝试 full scope 逆合成规划路径并成功返回路线。"
    else:
        state["retrosyn"] = degraded_route(smiles, "no-docker 模式默认不调用生产逆合成服务")
        mode = "降级模式"
        summary = "未调用生产逆合成服务，使用结构化降级路线继续流程。"
    state["history"].append("RETROSYN_PLANNED")
    finish_record(StageRecord(
        stage="逆合成规划",
        status="completed",
        mode=mode,
        summary=summary,
        details={"route_count": len(state["retrosyn"].get("routes", []))},
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["retrosyn"] = degraded_route(smiles, safe_error(exc))
    state["history"].append("RETROSYN_DEGRADED")
    finish_record(StageRecord(
        stage="逆合成规划",
        status="degraded",
        mode="降级模式",
        summary="生产逆合成路径不可用，已使用结构化降级路线。",
        details={"route_count": len(state["retrosyn"].get("routes", []))},
        error=safe_error(exc),
    ))

### 降级模式 逆合成规划
- 状态：`completed`
- 模式：`降级模式`
- 说明：未调用生产逆合成服务，使用结构化降级路线继续流程。

## 9. 供应可行性

无 Docker 模式下不保证 supply-oracle-svc 可用。此处基于逆合成路线中的 building blocks 生成结构化供应评估，标记为降级模式。

In [9]:
def route_building_blocks() -> list[dict[str, Any]]:
    routes = state.get("retrosyn", {}).get("routes", [])
    if not routes:
        return []
    route = routes[0]
    blocks = route.get("building_blocks") or []
    if blocks:
        return [block if isinstance(block, dict) else {"smiles": str(block)} for block in blocks]
    collected = []
    for step in route.get("steps", []):
        for item in step.get("building_blocks", []) or step.get("reactants", []) or []:
            collected.append(item if isinstance(item, dict) else {"smiles": str(item)})
    return collected

try:
    blocks = route_building_blocks()
    state["supply"] = {
        "status": "assessed",
        "degraded": True,
        "reason": "no-docker 模式未调用 supply-oracle-svc，使用路线 building blocks 做本机供应估计",
        "supply_assessment": {
            "total_blocks": len(blocks),
            "commercially_available": len(blocks),
            "avg_price_per_gram": 0.0,
            "avg_lead_time_days": 0.0,
            "supplier_diversity": 1 if blocks else 0,
            "overall_feasibility": "degraded_available" if blocks else "unavailable",
        },
        "block_assessments": [
            {"smiles": block.get("smiles", ""), "availability": "assumed_available", "degraded": True}
            for block in blocks
        ],
    }
    state["history"].append("SUPPLY_ASSESSED")
    finish_record(StageRecord(
        stage="供应可行性",
        status="completed",
        mode="降级模式",
        summary="未调用生产供应服务，基于降级路线中的 building blocks 生成供应可行性评估。",
        details=state["supply"]["supply_assessment"],
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["supply"] = {"status": "unavailable", "degraded": True, "error": safe_error(exc)}
    finish_record(StageRecord(
        stage="供应可行性",
        status="degraded",
        mode="降级模式",
        summary="供应评估失败，写入 unavailable 结果并继续流程。",
        error=safe_error(exc),
    ))

### 降级模式 供应可行性
- 状态：`completed`
- 模式：`降级模式`
- 说明：未调用生产供应服务，基于降级路线中的 building blocks 生成供应可行性评估。

## 10. SRB 合成方案

无 Docker 模式下不保证 SRB agent 的外部硬件/SiLA2 adapter 可用。此处把逆合成路线转换为结构化合成方案，并标记降级。

In [10]:
try:
    route = (state.get("retrosyn", {}).get("routes") or [{}])[0]
    steps = route.get("steps", [])
    state["srb"] = {
        "status": "compiled",
        "degraded": True,
        "reason": "no-docker 模式未调用生产 SRB/SiLA2 adapter，使用路线步骤生成结构化方案",
        "protocols": [
            {
                "protocol_id": "degraded-srb-protocol-1",
                "target_smiles": first_candidate_smiles(),
                "steps": [
                    {
                        "srb_step_id": f"srb-step-{idx + 1}",
                        "operation": step.get("operation", "synthesis_step"),
                        "reactants": step.get("reactants", []),
                        "degraded": True,
                    }
                    for idx, step in enumerate(steps)
                ],
            }
        ],
    }
    state["history"].append("SRB_COMPILED")
    finish_record(StageRecord(
        stage="SRB 合成方案",
        status="completed",
        mode="降级模式",
        summary="未调用生产 SRB/SiLA2 adapter，已生成结构化降级合成方案。",
        details={"protocol_count": len(state["srb"].get("protocols", []))},
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["srb"] = {"status": "unavailable", "degraded": True, "error": safe_error(exc)}
    finish_record(StageRecord(
        stage="SRB 合成方案",
        status="degraded",
        mode="降级模式",
        summary="SRB 合成方案失败，写入 unavailable 结果并继续流程。",
        error=safe_error(exc),
    ))

### 降级模式 SRB 合成方案
- 状态：`completed`
- 模式：`降级模式`
- 说明：未调用生产 SRB/SiLA2 adapter，已生成结构化降级合成方案。

## 11. Critic 审核

本阶段优先调用项目内 ScientificCriticAgent。若 critic 依赖不可用，则降级为基于 validation/supply 的最小审核结果。

In [11]:
def best_validation_row() -> dict[str, Any]:
    rows = [row for row in state.get("validation", {}).get("results", []) if isinstance(row, dict)]
    if not rows:
        return {}
    return max(rows, key=lambda row: float(row.get("admet_score") or row.get("composite_score") or 0.0))

try:
    from critic_agent.agent import ScientificCriticAgent

    state["critic"] = await ScientificCriticAgent().evaluate_molecule(
        {
            "project_id": REQUEST["project_id"],
            "run_id": REQUEST["run_id"],
            "smiles": first_candidate_smiles(),
            "properties": best_validation_row(),
        }
    )
    mode = "真实执行"
    summary = "已调用项目内 ScientificCriticAgent 对候选分子进行审核。"
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["critic"] = {
        "verdict": "review_required",
        "degraded": True,
        "reason": "critic agent 不可用，使用最小规则审核结果",
        "rule_results": [
            {"rule": "validation_passed", "passed": bool(state.get("validation", {}).get("passed"))},
            {"rule": "supply_available", "passed": state.get("supply", {}).get("supply_assessment", {}).get("overall_feasibility") != "unavailable"},
        ],
        "total_rules": 2,
        "error": safe_error(exc),
    }
    mode = "降级模式"
    summary = "Critic agent 不可用，已使用最小规则生成降级审核结果。"

state["history"].append("CRITIC_REVIEWED")
finish_record(StageRecord(
    stage="Critic 审核",
    status="completed",
    mode=mode,
    summary=summary,
    details={"verdict": state["critic"].get("verdict"), "total_rules": state["critic"].get("total_rules")},
))

### 降级模式 Critic 审核
- 状态：`completed`
- 模式：`降级模式`
- 说明：Critic agent 不可用，已使用最小规则生成降级审核结果。

StageRecord(stage='Critic 审核', status='completed', mode='降级模式', summary='Critic agent 不可用，已使用最小规则生成降级审核结果。', details={'verdict': 'review_required', 'total_rules': 2}, error=None, started_at='2026-06-10T17:20:35.833035+00:00', finished_at='2026-06-10T17:20:35.833049+00:00')

## 12. Provenance / Audit / CRG 存证

无 Docker 模式下不保证 Neo4j、MinIO、Provenance DB、Sigstore/Rekor 可用。  
本阶段默认生成本机内存 provenance/audit/CRG 记录，明确标记为降级模式。

In [12]:
try:
    artifact_id = f"artifact-{REQUEST['run_id']}-workflow-state"
    crg_beliefs = [
        {
            "id": f"belief-{idx}-{record.stage}",
            "subject": REQUEST["run_id"],
            "predicate": "workflow_stage",
            "object": record.stage,
            "confidence": 1.0,
            "source_agent": "notebook_no_docker_runner",
            "mode": record.mode,
        }
        for idx, record in enumerate(stage_records)
    ]
    state["crg"] = {
        "degraded": True,
        "beliefs": crg_beliefs,
        "edges": [
            {
                "source_belief_id": crg_beliefs[idx - 1]["id"],
                "target_belief_id": crg_beliefs[idx]["id"],
                "relation": "derives_from",
            }
            for idx in range(1, len(crg_beliefs))
        ],
    }
    state["provenance"] = {
        "recorded": True,
        "degraded": True,
        "artifact_id": artifact_id,
        "reason": "no-docker 模式未写入生产 Provenance DB / Neo4j / Sigstore/Rekor",
        "stage_count": len(stage_records),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    state["artifact_ids"].append(artifact_id)
    state["audit"] = {
        "run_id": REQUEST["run_id"],
        "trace_id": REQUEST["trace_id"],
        "records": [record.__dict__ for record in stage_records],
    }
    state["history"].append("PROVENANCE_RECORDED")
    finish_record(StageRecord(
        stage="Provenance / Audit / CRG 存证",
        status="completed",
        mode="降级模式",
        summary="未写入生产存证系统，已生成本机内存 provenance/audit/CRG 记录并返回。",
        details={"artifact_id": artifact_id, "belief_count": len(crg_beliefs)},
    ))
except Exception as exc:
    if not ALLOW_DEGRADED:
        raise
    state["provenance"] = {"recorded": False, "degraded": True, "error": safe_error(exc)}
    finish_record(StageRecord(
        stage="Provenance / Audit / CRG 存证",
        status="degraded",
        mode="降级模式",
        summary="本机存证记录生成失败，但流程会返回已有状态。",
        error=safe_error(exc),
    ))

### 降级模式 Provenance / Audit / CRG 存证
- 状态：`completed`
- 模式：`降级模式`
- 说明：未写入生产存证系统，已生成本机内存 provenance/audit/CRG 记录并返回。

## 13. 返回结果给用户

本阶段汇总所有状态，模拟 API 返回给前端/用户的最终响应。

In [13]:
real_stage_count = sum(1 for record in stage_records if record.mode == "真实执行")
degraded_stage_count = sum(1 for record in stage_records if record.mode == "降级模式")

response = {
    "run_id": REQUEST["run_id"],
    "trace_id": REQUEST["trace_id"],
    "status": "completed_with_degraded_steps" if any(r.mode == "降级模式" for r in stage_records) else "completed",
    "workflow_scope": REQUEST["workflow_scope"],
    "history": list(state.get("history", [])),
    "stage_summary": [
        {
            "stage": record.stage,
            "status": record.status,
            "mode": record.mode,
            "summary": record.summary,
        }
        for record in stage_records
    ],
    "candidates": state.get("candidates", []),
    "validation": state.get("validation", {}),
    "retrosyn": state.get("retrosyn", {}),
    "supply": state.get("supply", {}),
    "srb": state.get("srb", {}),
    "critic": state.get("critic", {}),
    "provenance": state.get("provenance", {}),
    "crg": state.get("crg", {}),
    "execution_mode_counts": {
        "真实执行": real_stage_count,
        "降级模式": degraded_stage_count,
    },
}

display(Markdown("# 最终结果"))
display(Markdown(f"- 运行状态：`{response['status']}`"))
display(Markdown(f"- 候选分子数量：`{len(response['candidates'])}`"))
display(Markdown(f"- 真实执行阶段数量：`{real_stage_count}`"))
display(Markdown(f"- 降级模式阶段数量：`{degraded_stage_count}`"))
display(Markdown(f"- 阶段数量：`{len(response['stage_summary'])}`"))
display(Markdown("## 阶段摘要"))
for item in response["stage_summary"]:
    display(Markdown(f"- **{item['stage']}**：`{item['status']}` / `{item['mode']}` - {item['summary']}"))

display(Markdown("## API 响应 JSON"))
display(Markdown("```json\n" + public_details(response)[:20000] + "\n```"))

# 最终结果

- 运行状态：`completed_with_degraded_steps`

- 候选分子数量：`4`

- 真实执行阶段数量：`3`

- 降级模式阶段数量：`6`

...

- **SRB 合成方案**：`completed` / `降级模式` - 未调用生产 SRB/SiLA2 adapter，已生成结构化降级合成方案。

- **Critic 审核**：`completed` / `降级模式` - Critic agent 不可用，已使用最小规则生成降级审核结果。

- **Provenance / Audit / CRG 存证**：`completed` / `降级模式` - 未写入生产存证系统，已生成本机内存 provenance/audit/CRG 记录并返回。

## API 响应 JSON

```json
{
  "run_id": "notebook-run-1781111714",
  "trace_id": "notebook-trace-1781111714",
  "status": "completed_with_degraded_steps",
  "workflow_scope": "no_docker_full_flow",
  "history": [
    "CIG_COMPILED",
    "GENERATED",
    "VALIDATED",
    "RETROSYN_PLANNED",
    "SUPPLY_ASSESSED",
    "SRB_COMPILED",
    "CRITIC_REVIEWED",
    "PROVENANCE_RECORDED"
  ],
  "stage_summary": [
    {
      "stage": "前端/API -> 自然语言设计意图",
      "status": "completed",
      "mode": "降级模式",
      "summary": "未启动浏览器和 API Gateway 服务；使用 notebook 内结构化 API payload 代表前端/API 提交。"
    },
    {
      "stage": "CIG 编译 -> HCIV / intent cone",
      "status": "completed",
      "mode": "真实执行",
      "summary": "已使用本地 CIG compiler 解析自然语言意图，并生成 HCIV 与 intent cone。"
    },
    {
      "stage": "生成器调度 -> 分子生成",
      "status": "completed",
      "mode": "真实执行",
      "summary": "未启动生产生成器服务；使用项目内本机 RDKitRandomGenerator 真实生成候选分子。"
    },
    {
      "stage": "Oracle 验证",
      "status": "completed",
      "mode": "真实执行",
      "summary": "未调用生产 Boltz/FEP Oracle；使用项目内本机 RDKit/L0 Oracle 与 MolPredictEngine 真实验证候选分子。"
    },
    {
      "stage": "逆合成规划",
      "status": "completed",
      "mode": "降级模式",
      "summary": "未调用生产逆合成服务，使用结构化降级路线继续流程。"
    },
    {
      "stage": "供应可行性",
      "status": "completed",
      "mode": "降级模式",
      "summary": "未调用生产供应服务，基于降级路线中的 building blocks 生成供应可行性评估。"
    },
    {
      "stage": "SRB 合成方案",
      "status": "completed",
      "mode": "降级模式",
      "summary": "未调用生产 SRB/SiLA2 adapter，已生成结构化降级合成方案。"
    },
    {
      "stage": "Critic 审核",
      "status": "completed",
      "mode": "降级模式",
      "summary": "Critic agent 不可用，已使用最小规则生成降级审核结果。"
    },
    {
      "stage": "Provenance / Audit / CRG 存证",
      "status": "completed",
      "mode": "降级模式",
      "summary": "未写入生产存证系统，已生成本机内存 provenance/audit/CRG 记录并返回。"
    }
  ],
  "candidates": [
    {
      "id": "",
      "smiles": "c1ccccc1",
      "canonical_smiles": "c1ccccc1",
      "generator_name": "rdkit_random",
      "humu_embedding": null,
      "properties": {},
      "embedding": null,
      "valid_rdkit_molecule": true
    },
    {
      "id": "",
      "smiles": "CC(N)Cc1ccc(C(C)C(=O)O)cc1",
      "canonical_smiles": "CC(N)Cc1ccc(C(C)C(=O)O)cc1",
      "generator_name": "rdkit_random",
      "humu_embedding": null,
      "properties": {},
      "embedding": null,
      "valid_rdkit_molecule": true
    },
    {
      "id": "",
      "smiles": "c1cnccn1",
      "canonical_smiles": "c1cnccn1",
      "generator_name": "rdkit_random",
      "humu_embedding": null,
      "properties": {},
      "embedding": null,
      "valid_rdkit_molecule": true
    },
    {
      "id": "",
      "smiles": "OOCl",
      "canonical_smiles": "OOCl",
      "generator_name": "rdkit_random",
      "humu_embedding": null,
      "properties": {},
      "embedding": null,
      "valid_rdkit_molecule": true
    }
  ],
  "validation": {
    "passed": true,
    "threshold": 0.0,
    "results": [
      {
        "smiles": "c1ccccc1",
        "canonical_smiles": "c1ccccc1",
        "valid": true,
        "molecular_weight": 78.11399999999999,
        "exact_mass": 78.046950192,
        "heavy_atoms": 6,
        "logp": 1.6866,
        "tpsa": 0.0,
        "hbd": 0,
        "hba": 0,
        "rotatable_bonds": 0,
        "aromatic_rings": 1,
        "rings": 1,
        "fraction_csp3": 0.0,
        "formal_charge": 0,
        "qed": 0.4426283718993647,
        "sa_score": 1.091,
        "lipinski_violations": 0,
        "formula": "C6H6",
        "drug_likeness": {
          "lipinski_pass": true,
          "veber_pass": true,
          "egan_pass": true,
          "qed_label": "moderate"
        },
        "admet": {
          "logd": 1.1866,
          "solubility_logS": -2.4618,
          "clearance_ml_min_kg": 0.5,
          "half_life_h": 9.4377,
          "bioavailability_pct": 70.0,
          "ppb_pct": 76.75,
          "herg_ic50_uM": 12.0,
          "caco2_logPapp": -4.5,
          "bbb_permeable": true,
          "pampa_high": true,
          "herg_risk": "low",
          "cyp3a4_substrate_likely": false
        },
        "humu_embedding_norm": 2.667357921600342,
        "humu_embedding_mean": 0.01591724529862404,
        "humu_embedding_dim": 129,
        "composite_score": 0.7456,
        "inchi_key": "UHOVQNZJYSORNB-UHFFFAOYSA-N",
        "device": "cpu",
        "error": null,
        "admet_score": 0.7281,
        "pains_alert": false,
        "pains_alerts": []
      },
      {
        "smiles": "CC(N)Cc1ccc(C(C)C(=O)O)cc1",
        "canonical_smiles": "CC(N)Cc1ccc(C(C)C(=O)O)cc1",
        "valid": true,
        "molecular_weight": 207.27299999999997,
        "exact_mass": 207.125928784,
        "heavy_atoms": 15,
        "logp": 1.7644,
        "tpsa": 63.32000000000001,
        "hbd": 2,
        "hba": 2,
        "rotatable_bonds": 4,
        "aromatic_rings": 1,
        "rings": 1,
        "fraction_csp3": 0.4166666666666667,
        "formal_charge": 0,
        "qed": 0.7904139492713904,
        "sa_score": 4.936,
        "lipinski_violations": 0,
        "formula": "C12H17NO2",
        "drug_likeness": {
          "lipinski_pass": true,
          "veber_pass": true,
          "egan_pass": true,
          "qed_label": "excellent"
        },
        "admet": {
          "logd": 1.2644,
          "solubility_logS": -3.8078,
          "clearance_ml_min_kg": 0.5,
          "half_life_h": 6.8545,
          "bioavailability_pct": 70.0,
          "ppb_pct": 77.06,
          "herg_ic50_uM": 12.0,
          "caco2_logPapp": -4.832,
          "bbb_permeable": true,
          "pampa_high": true,
          "herg_risk": "low",
          "cyp3a4_substrate_likely": false
        },
        "humu_embedding_norm": 2.318744421005249,
        "humu_embedding_mean": 0.023293061181902885,
        "humu_embedding_dim": 129,
        "composite_score": 0.7526,
        "inchi_key": "ITXPMTVOCAGCOQ-UHFFFAOYSA-N",
        "device": "cpu",
        "error": null,
        "admet_score": 0.6109,
        "pains_alert": false,
        "pains_alerts": []
      },
      {
        "smiles": "c1cnccn1",
        "canonical_smiles": "c1cnccn1",
        "valid": true,
        "molecular_weight": 80.09,
        "exact_mass": 80.037448128,
        "heavy_atoms": 6,
        "logp": 0.4765999999999999,
        "tpsa": 25.78,
        "hbd": 0,
        "hba": 2,
        "rotatable_bonds": 0,
        "aromatic_rings": 1,
        "rings": 1,
        "fraction_csp3": 0.0,
        "formal_charge": 0,
        "qed": 0.45255714309799316,
        "sa_score": 1.1,
        "lipinski_violations": 0,
        "formula": "C4H4N2",
        "drug_likeness": {
          "lipinski_pass": true,
          "veber_pass": true,
          "egan_pass": true,
          "qed_label": "moderate"
        },
        "admet": {
          "logd": -0.0234,
          "solubility_logS": -1.6345,
          "clearance_ml_min_kg": 0.5,
          "half_life_h": 9.3982,
          "bioavailability_pct": 70.0,
          "ppb_pct": 71.91,
          "herg_ic50_uM": 12.0,
          "caco2_logPapp": -4.5,
          "bbb_permeable": false,
          "pampa_high": false,
          "herg_risk": "low",
          "cyp3a4_substrate_likely": false
        },
        "humu_embedding_norm": 2.675138235092163,
        "humu_embedding_mean": 0.004299934487789869,
        "humu_embedding_dim": 129,
        "composite_score": 0.7498,
        "inchi_key": "KYQCOXFCLRTKLS-UHFFFAOYSA-N",
        "device": "cpu",
        "error": null,
        "admet_score": 0.7305,
        "pains_alert": false,
        "pains_alerts": []
      },
      {
        "smiles": "OOCl",
        "canonical_smiles": "OOCl",
        "valid": true,
        "molecular_weight": 68.459,
        "exact_mass": 67.966506952,
        "heavy_atoms": 3,
        "logp": 0.6297999999999999,
        "tpsa": 29.46,
        "hbd": 1,
        "hba": 2,
        "rotatable_bonds": 0,
        "aromatic_rings": 0,
        "rings": 0,
        "fraction_csp3": 0.0,
        "formal_charge": 0,
        "qed": 0.33326012219345186,
        "sa_score": 1.342,
        "lipinski_violations": 0,
        "formula": "HClO2",
        "drug_likeness": {
          "lipinski_pass": true,
          "veber_pass": true,
          "egan_pass": true,
          "qed_label": "poor"
        },
        "admet": {
          "logd": 0.1298,
          "solubility_logS": -1.6254,
          "clearance_ml_min_kg": 0.5,
          "half_life_h": 9.6308,
          "bioavailability_pct": 70.0,
          "ppb_pct": 72.52,
          "herg_ic50_uM": 12.0,
          "caco2_logPapp": -4.5,
          "bbb_permeable": false,
          "pampa_high": false,
          "herg_risk": "low",
          "cyp3a4_substrate_likely": false
        },
        "humu_embedding_norm": 2.191087245941162,
        "humu_embedding_mean": 0.021348142996430397,
        "humu_embedding_dim": 129,
        "composite_score": 0.6867,
        "inchi_key": "HSLBPQGVXMOSRA-UHFFFAOYSA-N",
        "device": "cpu",
        "error": null,
        "admet_score": 0.68,
        "pains_alert": false,
        "pains_alerts": []
      }
    ],
    "oracle_level": "L0_RDKit_local"
  },
  "retrosyn": {
    "status": "degraded",
    "degraded": true,
    "reason": "no-docker 模式默认不调用生产逆合成服务",
    "routes": [
      {
        "route_id": "degraded-route-1",
        "target_smiles": "c1ccccc1",
        "score": 0.0,
        "steps": [
          {
            "step_id": "step-1",
            "operation": "retrosynthesis_placeholder",
            "reactants": [
              "CCO",
              "O"
            ],
            "building_blocks": [
              {
                "smiles": "CCO"
              },
              {
                "smiles": "O"
              }
            ]
          }
        ],
        "building_blocks": [
          {
            "smiles": "CCO"
          },
          {
            "smiles": "O"
          }
        ]
      }
    ]
  },
  "supply": {
    "status": "assessed",
    "degraded": true,
    "reason": "no-docker 模式未调用 supply-oracle-svc，使用路线 building blocks 做本机供应估计",
    "supply_assessment": {
      "total_blocks": 2,
      "commercially_available": 2,
      "avg_price_per_gram": 0.0,
      "avg_lead_time_days": 0.0,
      "supplier_diversity": 1,
      "overall_feasibility": "degraded_available"
    },
    "block_assessments": [
      {
        "smiles": "CCO",
        "availability": "assumed_available",
        "degraded": true
      },
      {
        "smiles": "O",
        "availability": "assumed_available",
        "degraded": true
      }
    ]
  },
  "srb": {
    "status": "compiled",
    "degraded": true,
    "reason": "no-docker 模式未调用生产 SRB/SiLA2 adapter，使用路线步骤生成结构化方案",
    "protocols": [
      {
        "protocol_id": "degraded-srb-protocol-1",
        "target_smiles": "c1ccccc1",
        "steps": [
          {
            "srb_step_id": "srb-step-1",
            "operation": "retrosynthesis_placeholder",
            "reactants": [
              "CCO",
              "O"
            ],
            "degraded": true
          }
        ]
      }
    ]
  },
  "critic": {
    "verdict": "review_required",
    "degraded": true,
    "reason": "critic agent 不可用，使用最小规则审核结果",
    "rule_results": [
      {
        "rule": "validation_passed",
        "passed": true
      },
      {
        "rule": "supply_available",
        "passed": true
      }
    ],
    "total_rules": 2,
    "error": "ValueError: Failed to DNS resolve address 127.0.0.1:${NEO4J_BOLT_PORT}: [Errno -8] Servname not supported for ai_socktype"
  },
  "provenance": {
    "recorded": true,
    "degraded": true,
    "artifact_id": "artifact-notebook-run-1781111714-workflow-state",
    "reason": "no-docker 模式未写入生产 Provenance DB / Neo4j / Sigstore/Rekor",
    "stage_count": 8,
    "created_at": "2026-06-10T17:20:38.778155+00:00"
  },
  "crg": {
    "degraded": true,
    "beliefs": [
      {
        "id": "belief-0-前端/API -> 自然语言设计意图",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "前端/API -> 自然语言设计意图",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "降级模式"
      },
      {
        "id": "belief-1-CIG 编译 -> HCIV / intent cone",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "CIG 编译 -> HCIV / intent cone",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "真实执行"
      },
      {
        "id": "belief-2-生成器调度 -> 分子生成",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "生成器调度 -> 分子生成",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "真实执行"
      },
      {
        "id": "belief-3-Oracle 验证",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "Oracle 验证",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "真实执行"
      },
      {
        "id": "belief-4-逆合成规划",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "逆合成规划",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "降级模式"
      },
      {
        "id": "belief-5-供应可行性",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "供应可行性",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "降级模式"
      },
      {
        "id": "belief-6-SRB 合成方案",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "SRB 合成方案",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "降级模式"
      },
      {
        "id": "belief-7-Critic 审核",
        "subject": "notebook-run-1781111714",
        "predicate": "workflow_stage",
        "object": "Critic 审核",
        "confidence": 1.0,
        "source_agent": "notebook_no_docker_runner",
        "mode": "降级模式"
      }
    ],
    "edges": [
      {
        "source_belief_id": "belief-0-前端/API -> 自然语言设计意图",
        "target_belief_id": "belief-1-CIG 编译 -> HCIV / intent cone",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-1-CIG 编译 -> HCIV / intent cone",
        "target_belief_id": "belief-2-生成器调度 -> 分子生成",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-2-生成器调度 -> 分子生成",
        "target_belief_id": "belief-3-Oracle 验证",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-3-Oracle 验证",
        "target_belief_id": "belief-4-逆合成规划",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-4-逆合成规划",
        "target_belief_id": "belief-5-供应可行性",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-5-供应可行性",
        "target_belief_id": "belief-6-SRB 合成方案",
        "relation": "derives_from"
      },
      {
        "source_belief_id": "belief-6-SRB 合成方案",
        "target_belief_id": "belief-7-Critic 审核",
        "relation": "derives_from"
      }
    ]
  },
  "execution_mode_counts": {
    "真实执行": 3,
    "降级模式": 6
  }
}
```